# Colab run: Conditions 1, 2, 3 on a GPU

Runs the experiment for the chosen facts on a Colab GPU and saves everything to Google Drive:
`baseline.jsonl` (one scored row per fact, condition, wording, m), `hidden/*.npy` (hidden states),
`run.log`, `run_info.json` (GPU, library versions, commit), and the NAFTA worked example.

**Before you start**
1. Runtime → Change runtime type → **L4 GPU**.
2. The repo is private, so Colab needs a GitHub token: make a fine-grained token on GitHub
   (Settings → Developer settings → Fine-grained tokens) with access to **only this repo** and
   **Contents: Read-only**, then add it in Colab's **Secrets** panel (key icon, left sidebar) as
   `GITHUB_TOKEN` and switch on "Notebook access". Never paste the token into a cell.
3. Push the commit you want to run. This notebook clones `BRANCH` from GitHub.

**If the session disconnects:** run all cells again. `run_baseline` skips every row already in
`baseline.jsonl` on Drive, so it carries on where it stopped.

**When it finishes:** Runtime → Disconnect and delete runtime. Compute units are charged for
every minute the GPU is connected, not just while it computes.

## 1. Settings

In [ ]:
BRANCH = "main"                       # git branch to run
RUN_NAME = "colab-l4-run1"            # folder name on Drive; use a new name for a fresh run
# The midterm sample, picked from a closed-book screen of all 59 facts on the laptop (2026-10-28).
# Groups are final only after this run's own Condition 1.
PARAMETRIC = [                        # correct closed-book for all 3 wordings
    "nafta_start", "usmca_start", "water_treaty_year", "uri_month", "mexico_energy_reform",
    "russia_ukraine_invasion", "hemp_thc_limit", "tcja_salt_cap", "tcja_corporate_rate", "harvey_year",
]
EXTRACTIVE = [                        # correct for none of the 3 wordings
    "iran_war_start", "mexico_president_2024", "hamas_attack_date", "rubio_secretary_of_state",
    "carney_prime_minister", "mexico_top_partner_year", "baker_institute_director",
    # added 2026-10-28 (checked by Donovan); not in the laptop screen, so this run's Condition 1 decides
    "metals_tariff_2025", "mexico_canada_tariff_2025", "sheinbaum_took_office",
]
FACTS = PARAMETRIC + EXTRACTIVE
CONDITIONS = "1 2 3"                  # 1 closed-book, 2 clean RAG (m = 0), 3 dose (m = 1 to 5)
REPO_URL = "github.com/donovansappet168/comp345-counterfactual-rag.git"

## 2. Check the GPU
Should say L4 (or A100) with about 24 GB (or 40 GB). Llama 3.1 8B in bf16 needs about 16 GB.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 GPU"
assert torch.cuda.is_bf16_supported(), "This GPU has no bf16 (a T4?). Switch to L4 or A100."
print(torch.cuda.get_device_name(0))

## 3. Google Drive (results live here, so a disconnect loses nothing)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/comp345-crag/{RUN_NAME}"
!mkdir -p "{OUT}/hidden" "{OUT}/examples"
print(OUT)

## 4. Clone the repo and install the pinned versions
Same transformers and sentence-transformers versions as the laptop, so the chat template and generation defaults match. Colab's own CUDA build of torch is used.

In [ ]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/repo
!git clone -q --branch {BRANCH} https://{token}@{REPO_URL} /content/repo
del token
!cd /content/repo && git remote set-url origin https://{REPO_URL} && git log --oneline -1
!pip install -q transformers==5.20.2 sentence-transformers==6.2.0 accelerate==1.15.0

## 5. Point results/ at Drive and record the setup

In [ ]:
import json, subprocess, sys, datetime
!rm -rf /content/repo/results && ln -s "{OUT}" /content/repo/results
sys.path.insert(0, "/content/repo/src")
import transformers, sentence_transformers
from crag import config
info = {
    "started": datetime.datetime.now().isoformat(timespec="seconds"),
    "commit": subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip(),
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "generator": config.GENERATOR, "chat_date": config.CHAT_DATE,
    "max_new_tokens": config.MAX_NEW_TOKENS, "n_samples": config.N_SAMPLES, "seed": config.SEED,
    "facts": {"parametric": PARAMETRIC, "extractive": EXTRACTIVE},
    "conditions": CONDITIONS,
}
with open(f"{OUT}/run_info.json", "w") as f:
    json.dump(info, f, indent=2)
print(json.dumps(info, indent=2))
assert FACTS, "Fill in FACTS in cell 1"

## 6. Smoke test (about 3 minutes, includes the 16 GB download)
One RAG prompt (NAFTA, m = 1) with 2 samples and hidden states. Nothing is saved. Check the answer looks sensible and note the seconds: the full run is about 420 prompts.

In [ ]:
%%writefile /content/smoke.py
import time
import torch
from crag.facts import get
from crag.edit import build_context
from crag.generate import Llama
from crag.prompt import rag_prompt
from crag.run_baseline import top5

start = time.time()
llm = Llama()
print(f"model loaded in {time.time() - start:.0f}s on {llm.device}")
fact = get("nafta_start")
texts, slots = build_context(top5(fact, 0), fact, 0, 1)
start = time.time()
result = llm.answer(rag_prompt(fact["questions"][0], texts), n_samples=2, keep_hidden=True)
print(f"one prompt (greedy + 2 samples + hidden states): {time.time() - start:.0f}s")
print("edited slot:", slots, "| prompt tokens:", result["prompt_tokens"])
print("greedy:", result["greedy"])
print("samples:", result["samples"])
print("hidden:", result["hidden"].shape)
if torch.cuda.is_available():
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
!cd /content/repo && PYTHONPATH=src python /content/smoke.py 2>&1 | grep -v max_new_tokens

## 7. The run
Conditions 1, 2, 3 for every fact in `FACTS`: 3 wordings each, m = 0 to 5, 10 samples per prompt. Progress lines go to `run.log` on Drive too. Safe to re-run this cell after a disconnect.

In [ ]:
facts_arg = " ".join(FACTS)
!cd /content/repo && PYTHONPATH=src python -u -m crag.run_baseline --facts {facts_arg} --conditions {CONDITIONS} 2>&1 | grep --line-buffered -v max_new_tokens | tee -a "{OUT}/run.log"

## 8. The NAFTA worked example (7 prompts)
Written to the repo's `examples/`, then copied to Drive.

In [ ]:
!cd /content/repo && PYTHONPATH=src python -u -m crag.worked_example nafta_start --run 2>&1 | grep -v max_new_tokens | tail -3
!cp /content/repo/examples/nafta_start.md /content/repo/examples/nafta_start_outputs.json "{OUT}/examples/"

## 9. Quick check

In [ ]:
import pandas as pd
rows = pd.read_json(f"{OUT}/baseline.jsonl", lines=True)
print(len(rows), "rows;", len(list(__import__("pathlib").Path(OUT, "hidden").glob("*.npy"))), "hidden-state files")
print(rows.groupby(["condition", "m"], dropna=False)["label"].value_counts().unstack(fill_value=0))
info["finished"] = datetime.datetime.now().isoformat(timespec="seconds")
with open(f"{OUT}/run_info.json", "w") as f:
    json.dump(info, f, indent=2)

**Done? Runtime → Disconnect and delete runtime.** Then download the Drive folder into the repo's `results/` on the laptop to rebuild the demo notebook.